In [0]:
%sql
-- Preparing: catalog, schema, table
CREATE CATALOG IF NOT EXISTS demo_governance;
USE CATALOG demo_governance;
CREATE SCHEMA IF NOT EXISTS sales;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sales.transactions (
  transaction_id STRING,
  customer_name STRING,
  customer_email STRING,
  amount DOUBLE,
  country STRING
);


INSERT INTO sales.transactions VALUES 
('123', 'John', 'John@gmail.com', 10, 'USA')

In [0]:
%sql
INSERT INTO sales.transactions VALUES 
('322', 'Kate', 'Kate@gmail.com', 11, 'USA')

### for presentation purpose

In [0]:
%sql
ALTER TABLE demo_governance.sales.transactions SET OWNER TO `your_email`;

In [0]:
%sql
ALTER TABLE demo_governance.sales.transactions OWNER TO owners

In [0]:
%sql
SELECT * FROM demo_governance.sales.transactions

In [0]:
%sql
SHOW GRANTS ON SCHEMA demo_governance.sales;

In [0]:
%sql
SHOW GRANTS ON TABLE demo_governance.sales.transactions;

### RBAC

In [0]:
%sql

-- (RBAC = privileges for role)
-- GRANT USE CATALOG ON CATALOG demo_governance TO `analysts`;
-- GRANT USE SCHEMA ON SCHEMA demo_governance.sales TO `analysts`;
-- GRANT SELECT ON TABLE demo_governance.sales.transactions TO `analysts`;

-- All privileges
-- GRANT ALL PRIVILEGES ON TABLE demo_governance.sales.transactions TO `analysts`;
GRANT ALL PRIVILEGES ON TABLE demo_governance.sales.transactions TO `data_engineers`;


In [0]:
%sql
-- Odebranie uprawnień
REVOKE SELECT ON TABLE demo_governance.sales.transactions FROM `analysts`;

In [0]:
%sql
DELETE FROM sales.transactions
WHERE transaction_id = '322'

### ABAC

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_email(val STRING)
RETURNS STRING
RETURN IF(is_member('data_engineers'), val, '***@gmail.com');

ALTER TABLE demo_governance.sales.transactions
ALTER COLUMN customer_email
SET MASK mask_email;

In [0]:
%sql
-- Test
SELECT * FROM demo_governance.sales.transactions;

### TAGS

In [0]:
%sql
ALTER TABLE sales.transactions
ALTER COLUMN customer_email SET TAGS ('pii_type' = 'email');

In [0]:
pii_columns = spark.sql("""
SELECT
  catalog_name,   
  schema_name,
  table_name, 
  column_name 
FROM system.information_schema.column_tags
WHERE (tag_name = 'pii_type' AND tag_value = 'email')   
""")


In [0]:
display(pii_columns)

In [0]:
for row in pii_columns.collect():   

    sql = f"""
    ALTER TABLE {row.catalog_name}.{row.schema_name}.{row.table_name}
    ALTER COLUMN {row.column_name}
    SET MASK mask_email
    """

    print(sql)

In [0]:
%sql
SHOW GROUPS WITH USER `your_email`;